# Dimensiones y Hechos Capa Silver



####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.7 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 4.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: 7a5065b3-ff95-44b2-b153-6f674ffbdd43
Applying the following default arguments:
--glue_kernel_version 1.0.7
--enable-glue-datacatalog true
Waiting for session 7a5065b3-ff95-44b2-b153-6f674ffbdd43 to get into ready status...
Session 7a5065b3-ff95-44b2-b153-6f674ffbdd43 ha

#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [6]:
DB_CURATED = "final_db_bronze" 
TABLE_CURATED = "curated_us_accidents"   

dyf = glueContext.create_dynamic_frame.from_catalog(
    database=DB_CURATED,
    table_name=TABLE_CURATED
)
df = dyf.toDF()

df.printSchema()
print("Filas:", df.count(), "Columnas:", len(df.columns))


root
 |-- severity: integer (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- weather_timestamp: timestamp (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- temperature_f: double (nullable = true)
 |-- humidity_pct: double (nullable = true)
 |-- pressure_in: double (nullable = true)
 |-- visibility_mi: double (nullable = true)
 |-- wind_speed_mph: double (nullable = true)
 |-- precipitation_in: double (nullable = true)
 |-- wind_chill_f: double (nullable = true)

Filas: 500000 Columnas: 14


#### Atributos de Tiempo para dim_time


In [7]:
from pyspark.sql.functions import col, to_date, year, month, dayofmonth, hour, date_format

df_t = df.withColumn("date", to_date(col("start_time"))) \
         .withColumn("year", year(col("start_time"))) \
         .withColumn("month", month(col("start_time"))) \
         .withColumn("day", dayofmonth(col("start_time"))) \
         .withColumn("hour", hour(col("start_time"))) \
         .withColumn("day_name", date_format(col("start_time"), "EEEE"))


#### Dim Time 


In [8]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

dim_time = df_t.select("date","year","month","day","hour","day_name") \
               .dropna(subset=["date","hour"]) \
               .dropDuplicates()

w_time = Window.orderBy("date","hour")
dim_time = dim_time.withColumn("time_sk", dense_rank().over(w_time))

dim_time.show(5, truncate=False)
print("dim_time:", dim_time.count())


+----------+----+-----+---+----+--------+-------+
|date      |year|month|day|hour|day_name|time_sk|
+----------+----+-----+---+----+--------+-------+
|2016-01-14|2016|1    |14 |20  |Thursday|1      |
|2016-02-08|2016|2    |8  |0   |Monday  |2      |
|2016-02-09|2016|2    |9  |5   |Tuesday |3      |
|2016-02-09|2016|2    |9  |6   |Tuesday |4      |
|2016-02-09|2016|2    |9  |7   |Tuesday |5      |
+----------+----+-----+---+----+--------+-------+
only showing top 5 rows

dim_time: 53926


#### Dim Location

In [9]:
dim_location = df.select("state","city","timezone") \
                 .dropna(subset=["state","city","timezone"]) \
                 .dropDuplicates()

w_loc = Window.orderBy("state","city","timezone")
dim_location = dim_location.withColumn("location_sk", dense_rank().over(w_loc))

dim_location.show(5, truncate=False)
print("dim_location:", dim_location.count())


+-----+-----------+----------+-----------+
|state|city       |timezone  |location_sk|
+-----+-----------+----------+-----------+
|AL   |Abbeville  |US/Central|1          |
|AL   |Adamsville |US/Central|2          |
|AL   |Addison    |US/Central|3          |
|AL   |Alabaster  |US/Central|4          |
|AL   |Albertville|US/Central|5          |
+-----+-----------+----------+-----------+
only showing top 5 rows

dim_location: 13000


#### Dim Clima

In [10]:
weather_cols = [
    "weather_condition",
    "temperature_f","humidity_pct","pressure_in","visibility_mi",
    "wind_speed_mph","precipitation_in","wind_chill_f"
]

dim_weather = df.select(*weather_cols).dropDuplicates()

w_w = Window.orderBy(*weather_cols)
dim_weather = dim_weather.withColumn("weather_sk", dense_rank().over(w_w))

dim_weather.show(5, truncate=False)
print("dim_weather:", dim_weather.count())


+-----------------+-------------+------------+-----------+-------------+--------------+----------------+------------+----------+
|weather_condition|temperature_f|humidity_pct|pressure_in|visibility_mi|wind_speed_mph|precipitation_in|wind_chill_f|weather_sk|
+-----------------+-------------+------------+-----------+-------------+--------------+----------------+------------+----------+
|null             |null         |null        |null       |null         |null          |null            |null        |1         |
|null             |null         |null        |null       |null         |null          |0.0             |null        |2         |
|null             |null         |null        |null       |null         |null          |0.01            |null        |3         |
|null             |null         |null        |null       |null         |5.0           |0.0             |null        |4         |
|null             |null         |null        |null       |null         |6.0           |0.0       

#### Fact Accidentes

In [11]:
from pyspark.sql.functions import lit

fact_base = df_t.select(
    "severity",
    "state","city","timezone",
    "weather_condition","temperature_f","humidity_pct","pressure_in","visibility_mi",
    "wind_speed_mph","precipitation_in","wind_chill_f",
    "date","year","month","day","hour","day_name"
)


In [12]:
fact = fact_base.join(dim_time, on=["date","year","month","day","hour","day_name"], how="left") \
                .join(dim_location, on=["state","city","timezone"], how="left") \
                .join(dim_weather, on=weather_cols, how="left")


In [13]:
from pyspark.sql.functions import col

fact_accidents = fact.select(
    col("time_sk"),
    col("location_sk"),
    col("weather_sk"),
    col("severity").cast("int").alias("severity"),
    lit(1).alias("accident_cnt")
)

fact_accidents.show(5, truncate=False)
print("fact_accidents:", fact_accidents.count())


+-------+-----------+----------+--------+------------+
|time_sk|location_sk|weather_sk|severity|accident_cnt|
+-------+-----------+----------+--------+------------+
|16604  |2508       |null      |2       |1           |
|29137  |10563      |360731    |2       |1           |
|17416  |8798       |null      |3       |1           |
|20416  |923        |null      |4       |1           |
|48528  |1769       |256559    |2       |1           |
+-------+-----------+----------+--------+------------+
only showing top 5 rows

fact_accidents: 500000


In [14]:
print("Null time_sk:", fact_accidents.filter(col("time_sk").isNull()).count())
print("Null location_sk:", fact_accidents.filter(col("location_sk").isNull()).count())
print("Null weather_sk:", fact_accidents.filter(col("weather_sk").isNull()).count())


Null time_sk: 0
Null location_sk: 526
Null weather_sk: 161109


In [15]:
SILVER_BASE = "s3://ef-sin-bucket/silver/"

dim_time.write.mode("overwrite").parquet(SILVER_BASE + "silver_dim_time/")
dim_location.write.mode("overwrite").parquet(SILVER_BASE + "silver_dim_location/")
dim_weather.write.mode("overwrite").parquet(SILVER_BASE + "silver_dim_weather/")
fact_accidents.write.mode("overwrite").parquet(SILVER_BASE + "silver_fact_accidents/")
